In [2]:
import numpy as np 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from exotics.engine import ExoticOptions
from exotics.greeks import Greeks

In [3]:
r, q = 0.02, 0.0
K1, K2 = 90.0, 110.0 
T = 0.5 
S_grid = np.linspace(70, 130, 400)

In [ ]:
# Flat Volatility (20%) for reference pricing 
sigma_flat = 0.20
engine_flat = ExoticOptions(S_grid, r, q, sigma_flat)

In [ ]:
#Volatility skew parameters (implying higher vol for K1 than K2)
sigma_k1_skew = 0.25 
sigma_k2_skew = 0.15

In [6]:
price_flat = engine_flat.price_double_digital(K1, K2, T)
price_skew = engine_flat.price_double_digital(K1, K2, T, sigma_k1=sigma_k1_skew, sigma_k2=sigma_k2_skew)

fig_price = go.Figure()
fig_price.add_trace(go.Scatter(x=S_grid, y=price_flat, name='Price (Flat Volatility)', line=dict(color='black', dash='dash')))
fig_price.add_trace(go.Scatter(x=S_grid, y=price_skew, name='Price (With Volatility Skew)', line=dict(color='blue', width=3)))
fig_price.add_vline(x=K1, line_dash="dot", line_color="grey")
fig_price.add_vline(x=K2, line_dash="dot", line_color="grey")

fig_price.update_layout(
    title="<b>Pricing Double Digital</b>",
    xaxis_title="Spot", yaxis_title="Option Price", template="plotly_white", hovermode="x unified"
)
fig_price.show()

In [9]:
greeks = Greeks(engine_flat)

delta_dd = greeks.double_digital_delta(K1, K2, T=0.1) 
gamma_dd = greeks.double_digital_gamma(K1, K2, T=0.1)
vega_dd = greeks.double_digital_vega(K1, K2, T=0.1)

fig_greeks = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                           subplot_titles=("<b>Delta</b> ", 
                                           "<b>Gamma</b> ", 
                                           "<b>Vega</b> "))

fig_greeks.add_trace(go.Scatter(x=S_grid, y=delta_dd, name="Delta", line=dict(color='green')), row=1, col=1)
fig_greeks.add_trace(go.Scatter(x=S_grid, y=gamma_dd, name="Gamma", line=dict(color='red')), row=2, col=1)
fig_greeks.add_trace(go.Scatter(x=S_grid, y=vega_dd, name="Vega", line=dict(color='blue')), row=3, col=1)

for i in range(1, 4):
    fig_greeks.add_vline(x=K1, line_dash="dot", line_color="grey", row=i, col=1)
    fig_greeks.add_vline(x=K2, line_dash="dot", line_color="grey", row=i, col=1)
    fig_greeks.add_hline(y=0, line_color="black", line_width=1, row=i, col=1)

fig_greeks.update_layout(height=800, template="plotly_white")
fig_greeks.show()